In [3]:
import pandas as pd
import numpy as np

In [4]:
columns = [
    'variance',
    'skewness',
    'curtosis',
    'entropy',
    'label'
]

train_data = pd.read_csv('bank-note/train.csv', names=columns, header=None)
test_data = pd.read_csv('bank-note/test.csv', names=columns, header=None)


x_train = train_data.drop('label', axis=1).values
y_train = train_data['label'].values

x_test = test_data.drop('label', axis=1).values
y_test = test_data['label'].values

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Define the objective function for logistic regression with Gaussian prior
def objective_function(X, y, w, v):
    n = len(y)
    y_pred = sigmoid(X.dot(w))
    likelihood = -np.sum(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred)) / n
    prior = np.sum(w ** 2) / (2 * v)
    return likelihood + prior

#gradient of the odjective
def gradient(X, y, w, v):
    n = 1
    y_pred = sigmoid(X.dot(w))
    grad_likelihood = X.T.dot(y_pred - y) / n
    grad_prior = w / v
    return grad_likelihood + grad_prior



T = 100 
learning_rate = 0.1
d = 100 


y_train = np.array(y_train)
y_test = np.array(y_test)

prior_variances = [0.01, 0.1, 0.5, 1, 3, 5, 10, 100]

for v in prior_variances:
    np.random.seed(42)
    
    #initialize weights
    w = np.zeros(x_train.shape[1])  
    
    train_errors = []
    test_errors = []
    
    for epoch in range(T):
        
        # Shuffle time
        indices = np.random.permutation(len(y_train))
        X_train_shuffled = x_train[indices]
        y_train_shuffled = y_train[indices]
        
        gam = learning_rate / (1 + (learning_rate / d) * epoch)
        
        #stochastic gradient descent
        for i in range(len(y_train)):
            grad = gradient(X_train_shuffled[i], y_train_shuffled[i], w, v)
            w -= gam * grad

       #calculating error and acruacy
        train_pred = sigmoid(x_train.dot(w))
        train_pred_labels = (train_pred >= 0.5).astype(int)
        train_accuracy = np.mean(train_pred_labels == y_train)
        train_errors.append(1 - train_accuracy)  
        

        # Calculate test accuracy
        test_pred = sigmoid(x_test.dot(w))
        test_pred_labels = (test_pred >= 0.5).astype(int)
        test_accuracy = np.mean(test_pred_labels == y_test)
        test_errors.append(1 - test_accuracy)

    avg_train_error = np.mean(train_errors)
    avg_test_error = np.mean(test_errors)

    print(f"For v={v}, Average Train Error = {avg_train_error}, Average Test Error = {avg_test_error}")

/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_85105/3585250705.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_85105/3585250705.py:17: RuntimeWarning: overflow encountered in divide
  grad_prior = w / v
/var/folders/h0/f9d5nsss0s14zm26734cgfg80000gn/T/ipykernel_85105/3585250705.py:53: RuntimeWarning: invalid value encountered in subtract
  w -= gam * grad


For v=0.01, Average Train Error = 0.4461009174311928, Average Test Error = 0.4420000000000001
For v=0.1, Average Train Error = 0.39784403669724777, Average Test Error = 0.39292
For v=0.5, Average Train Error = 0.2783142201834862, Average Test Error = 0.28184
For v=1, Average Train Error = 0.2243348623853211, Average Test Error = 0.22932000000000002
For v=3, Average Train Error = 0.15606651376146785, Average Test Error = 0.16440000000000002
For v=5, Average Train Error = 0.13153669724770642, Average Test Error = 0.14134
For v=10, Average Train Error = 0.1010435779816514, Average Test Error = 0.11305999999999998
For v=100, Average Train Error = 0.06827981651376146, Average Test Error = 0.07833999999999999


In [6]:
# Remove the prior-related parts from the objective function
def objective_function(X, y, w):
    n = len(y)
    y_pred = sigmoid(X.dot(w))
    likelihood = -np.sum(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred)) / n
    return likelihood


def gradient(X, y, w):
    n = 1
    y_pred = sigmoid(X.dot(w))
    grad_likelihood = X.T.dot(y_pred - y) / n
    return grad_likelihood


# ML estimation doesn't use prior variance
for v in prior_variances:#[0]:
    np.random.seed(42)
    w = np.zeros(x_train.shape[1])  # Initialize model parameters

    train_errors = []
    test_errors = []

    #basically the same as before
    for epoch in range(T):
        # Shuffle
        indices = np.random.permutation(len(y_train))
        X_train_shuffled = x_train[indices]
        y_train_shuffled = y_train[indices]

        # Update learning rate for this epoch
        gamma_t = learning_rate / (1 + (learning_rate / d) * epoch)

        # Perform stochastic gradient descent for ML estimation
        for i in range(len(y_train)):
            grad = gradient(X_train_shuffled[i], y_train_shuffled[i], w)
            w -= gamma_t * grad

        # Calculate ERR
        train_pred = sigmoid(x_train.dot(w))
        train_pred_labels = (train_pred >= 0.5).astype(int)
        train_accuracy = np.mean(train_pred_labels == y_train)
        train_errors.append(1 - train_accuracy)

        test_pred = sigmoid(x_test.dot(w))
        test_pred_labels = (test_pred >= 0.5).astype(int)
        test_accuracy = np.mean(test_pred_labels == y_test)
        test_errors.append(1 - test_accuracy)

    avg_train_error = np.mean(train_errors)
    avg_test_error = np.mean(test_errors)
    print(f"For For ML estimation v={v}, Average Train Error = {avg_train_error}, Average Test Error = {avg_test_error}")

For For ML estimation v=0.01, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=0.1, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=0.5, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=1, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=3, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=5, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=10, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
For For ML estimation v=100, Average Train Error = 0.053474770642201845, Average Test Error = 0.06404
